In [2]:
#%pip install numpy
import numpy as np
import math
import xml.etree.ElementTree as ET

In [3]:

def precision(re:list):
    """Calcula la precisión como la proporción de elementos relevantes
    recuperados sobre el total de elementos recuperados.
    Esto sobre una lista de relevancia binaria (1 para relevante, 0 para no relevante).
    Tiene en cuenta que la lista puede estar vacía,
    en cuyo caso devuelve 0.0. Evitando asi la división por cero.
    --
    Parameters:
    re: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).

    Returns:
    float
        La precisión calculada.
    """
    result = 0.0
    size = len(re)
    if size > 0:
        relevance =  np.array(re)
        result = (np.sum(relevance == 1))/size
    return result

def precision_at_k(re:list, k:int):
    """Calcula la precisión en los primeros k elementos de la lista de relevancia binaria.
    
    Parameters:
    re: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).
    k: int
        Número de elementos a considerar para el cálculo de la precisión.
    
    Returns:
    float
        La precisión calculada en los primeros k elementos.
        Es la fracción de elementos devueltos relevantes entre los k primeros.   
        Esta metrica aumenta cuando los elementos relevantes se encuentran en las primeras posiciones. Y disminuye cuando los elementos relevantes se encuentran en las últimas posiciones.
    """
    result = 0.0
    size = len(re)
    if size > 0 and k > 0 and size >= k:
        relevance = np.array(re)
        result = np.sum(relevance[:k] ==1)/k
    return result        

def recall_at_k(relevance_query:list, number_relevant_docs:int, k:int):
    """Calcula el recall en los primeros k elementos de la lista de relevancia binaria.

    Parameters:
    relevance_query: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).
    number_relevant_docs: int
        Número total de documentos relevantes. Este valor R(q) está dado por un conjunto de referencia de relevancia para la consulta q.
    k: int
        Número de elementos a considerar para el cálculo del recall.

    Returns:
    float
        El recall calculado en los primeros k elementos.
        Es la fracción de elementos relevantes recuperados en k sobre el total de elementos relevantes R(q).
        Esta metrica puede subir o mantenerse constante. No puede disminuir.
    """
    result = 0.0
    size = len(relevance_query)
    if size > 0 and number_relevant_docs > 0 and k > 0 and size >= k:
        relevance = np.array(relevance_query)
        result = np.sum(relevance[:k] == 1)/number_relevant_docs
    return result


In [4]:
def average_precision(re: list):
    """Calcula la precisión promedio (Average Precision) para una lista de relevancia binaria.
    La precisión promedio es la media de las precisiones calculadas en cada posición k donde hay un elemento relevante.
    --
    Parameters:
    re: list
        Lista de relevancia binaria (1 para relevante, 0 para no relevante).

    Returns:
    float
        La precisión promedio calculada.
    """
    result = 0.0
    relevance = np.array(re)
    precisions=[]
    relevant_count = 0
    if len(relevance) > 0 and np.sum(relevance ==1) > 0 : # aqui evalua que haya al menos un elemento relevante en la lista de relevancia, asi, suma uno a la cuenta de elementos relevantes y calcula la precision en cada posicion k donde hay un elemento relevante.
        for k, value in enumerate(relevance):
            if  value==1:
                relevant_count += 1
                precisions.append(relevant_count/(k+1))
        result = sum(precisions)/len(precisions)
    return result

def mean_average_precision(relevance_queries:list[list]):
    """"Calcula la precisión promedio media (Mean Average Precision) para un conjunto de consultas.
    La precisión promedio media es la media de las precisiones promedio calculadas para cada consulta.
    
    Parameters:
    relevance_queries: list[list]
        Lista de listas de relevancia binaria (1 para relevante, 0 para no relevante) para cada consulta.
    
    Returns:
    float
        La precisión promedio media calculada.
        """
    precisions=[]
    result=0.0
    if len(relevance_queries) > 0:
        precisions = [average_precision(query) for query in relevance_queries]
        result = sum(precisions)/len(precisions)
    return result


In [5]:
def dcg_at_k(relevance_query: list, k: int, gain: str):
    """Calcula el la ganancia acumulada descontada (DCG) en los primeros k elementos de la lista de relevancia.
    Esta metrica tiene en cuenta la relevancia de los elementos y su posición en la lista.
    Los elementos relevantes en posiciones más altas contribuyen más a la ganancia acumulada.
    Las posiciones bajas castigan la ganancia acumulada.
    
    Parameters:
    relevance_query: list
        Lista de relevancia en escala particular no binaria.
    k: int
        Número de elementos a considerar.
    gain: str
        Tipo de ganancia a utilizar ('linear' o 'exponential').
        linear: La ganancia es proporcional a la relevancia del elemento.
        exponential: La ganancia es exponencial a la relevancia del elemento. 

    returns:
    float
        La ganancia acumulada descontada calculada en los primeros k elementos.    
    """
    relevance = np.array(relevance_query)
    size = len(relevance)
    result = 0.0
    if size > 0 and k > 0 and size >= k:
        for i, rel in enumerate(relevance[:k]):
            if gain == 'linear':
                result += rel / math.log2(i + 2)
            elif gain == 'exponential':
                result += (2 ** rel - 1) / math.log2(i + 2)
    return result

def ndcg_at_k(relevance_query, k, gain='exponential'):
    """Calcula la ganancia acumulada descontada normalizada (NDCG) en los primeros k elementos de la lista de relevancia.
    Esta metrica usa como referencia la ganancia acumulada descontada ideal (IDCG) que se calcula ordenando la lista de relevancia de mayor a menor.
    La NDCG se calcula como la relación entre la DCG y la IDCG.
    
    Parameters:
    relevance_query: list
        Lista de relevancia en escala particular no binaria.
    k: int
        Número de elementos a considerar.
    gain: str
        Tipo de ganancia a utilizar ('linear' o 'exponential').
        linear: La ganancia es proporcional a la relevancia del elemento.
        exponential: La ganancia es exponencial a la relevancia del elemento.   
        
        returns:
    float
        La ganancia acumulada descontada normalizada calculada en los primeros k elementos.
    """
    result = 0.0
    if len(relevance_query)>0 : 
        ideal_query = sorted(relevance_query, reverse=True)
        dcg_Ideal = dcg_at_k(ideal_query,k,gain)
        if dcg_Ideal > 0 :
            result = (dcg_at_k(relevance_query,k,gain))/(dcg_Ideal)
    return result    


In [6]:
relevance_query = [3, 2, 3, 0, 1, 2, 3, 0, 0, 1]
k = 5
ndcg_at_k(relevance_query, k, gain='linear')
ndcg_at_k(relevance_query, k, gain='exponential')

np.float64(0.7357689654680096)